In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from astropy.visualization import time_support
from astropy.time import Time
import astropy.units as u

from sunpy import timeseries as ts
from sunpy.net import Fido
from sunpy.net import attrs as a

from stixpy.net.client import STIXClient
from stixpy.timeseries import quicklook 

import datetime as dt
from sunpy.time import parse_time
from sunpy.time import TimeRange

import pandas as pd

In [1]:
def rank_overlaps(start, end, sci_query):
    overlaps=[]
    for n in range(len(sci_query[0])):
        latest_start = max(start, sci_query[0][n][0].datetime)
        earliest_end = min(end, sci_query[0][n][1].datetime)
        
        overlap = earliest_end - latest_start
        overlaps.append((overlap.total_seconds(), n))
        
    return sorted(overlaps, key=lambda x: x[0], reverse=True)

In [ ]:
df_list2=[]
indices2=[]
for ind in top100.index:
    sci_query = Fido.search(a.Time(top100['start_UTC'].loc[ind], 
                                   top100['end_UTC'].loc[ind]), 
                        a.Instrument.stix,
                        a.stix.DataType.sci)
    sci_query['stix'].filter_for_latest_version()

    start = parse_time(top100['start_UTC'].loc[ind]).datetime
    end = parse_time(top100['end_UTC'].loc[ind]).datetime

    overlaps_ranked = rank_overlaps(start, end, sci_query)
    
    badrange=False
    data=False
    for overlap in overlaps_ranked:
        sci_files = Fido.fetch(sci_query[0][overlap])
        sci_data = Product(sci_files)
    
        if sci_data.energies["e_high"][len(sci_data.energies["e_high"])-1]<100*u.keV or sci_data.energies["e_low"][0]>25*u.keV:
            badrange=True
        
        if not badrange:
            sci_df = get_stix_df(sci_data, energy_ranges)
            df_list2.append(sci_df)
            indices2.append(ind)
            break

In [ ]:
fig, axes = plt.subplots(nrows=20, ncols=5, figsize=(15, 50))
axes = axes.flatten()
i=0
for ind in top100.index:
    ax=axes[i]
    ax.set_ylabel('ct/(keV s)')
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    ax.set_title(f"{parse_time(top100['start_UTC'].loc[ind]).datetime.year}-{parse_time(top100['start_UTC'].loc[ind]).datetime.month}")
    if ind in indices2:
        sci_df = df_list2[np.where(indices2_array==ind)[1][0]]
        
        start = parse_time(top100['start_UTC'].loc[ind]).datetime
        end = parse_time(top100['end_UTC'].loc[ind]).datetime
        
        plot_df = sci_df.truncate(start, end)
            
        plot_df.index = parse_time(plot_df.index).datetime
        
        sci_df.plot(ax=ax, y='25-50 keV', label='25-50 keV', legend=None)
        sci_df.plot(ax=ax, y='50-84 keV', label='50-84 keV', legend=None)
        
    else:
        ax.plot([0,1],[0,1], color='red')

    i+=1

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=2, fontsize=11)
fig.subplots_adjust(hspace=1.5)
fig.subplots_adjust(wspace=0.5)

plt.show()
fig.savefig("C:/Users/derva/OneDrive/Documents/DIAS/solar-orbiter-flare-statistics/figures/top_100_flares_science_data_full.png", bbox_inches='tight')